In [ ]:
"""
Knowledge ingestion pipeline.

Loads documents from data/ directory, splits into chunks,
generates embeddings, and saves the index to disk.

Usage: python ingest.py
"""


def ingest():
    # TODO:
    # 1. Load documents from config.data_dir (PDF, TXT, MD)
    # 2. Split into chunks using TextSplitter
    # 3. Generate embeddings
    # 4. Build vector store (FAISS, Qdrant, Chroma, etc.)
    # 5. Save index to config.index_dir
    # 6. Save chunks for BM25 retriever (pickle or JSON)
    pass


if __name__ == "__main__":
    ingest()


In [35]:
# Document Loaders - LlamaIndex - SimpleDirectoryReader

from llama_index.core import SimpleDirectoryReader

from config import Settings
settings = Settings()

documents = SimpleDirectoryReader(
    input_dir = settings.data_dir, 
    recursive=True,       #вкладені підпапки
    filename_as_id=True   #назву файлу як унікальний ідентифікатор (doc_id)
    ).load_data()

print(f"📄 Loaded {len(documents)} document fragments")
total_chars = sum(len(doc.text) for doc in documents)
print(f"Total_chars: {total_chars}")
print(f"\nFirst doc metadata: {documents[0].metadata}")
print(f"\nFirst doc metadata: {documents[10].metadata}")
print(f"First 200 chars: {documents[0].text[:200]}...")

📄 Loaded 52 document fragments
Total_chars: 173207

First doc metadata: {'page_label': '1', 'file_name': 'langchain.pdf', 'file_path': 'c:\\Users\\User\\Desktop\\robotdream-course\\agents\\homework-lesson-5\\data\\langchain.pdf', 'file_type': 'application/pdf', 'file_size': 229019, 'creation_date': '2026-03-19', 'last_modified_date': '2026-03-18'}

First doc metadata: {'page_label': '7', 'file_name': 'large-language-model.pdf', 'file_path': 'c:\\Users\\User\\Desktop\\robotdream-course\\agents\\homework-lesson-5\\data\\large-language-model.pdf', 'file_type': 'application/pdf', 'file_size': 1173003, 'creation_date': '2026-03-19', 'last_modified_date': '2026-03-18'}
First 200 chars: LangChain
Developer Harrison Chase
Initial release October 2022
Stable release0.1.16[1] / 11 April 2024
Written in Python and JavaScript
Type Software framework for largelanguage model applicationdeve...


In [14]:
# Chunking - Recursive по абзацу/реченню - RecursiveCharacterTextSplitter

from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=150, 
    separators=["\n\n", "\n", ".", "!", "?", ",", " ", ""]
)

langchain_docs = []
for doc in documents:
    chunks = recursive_splitter.split_text(doc.text)
    for chunk in chunks:
        langchain_docs.append({
            "text": chunk,
            "metadata": doc.metadata    # Варіант із збереженням метаданих
        })

print(f"📄 Total chunks created: {len(langchain_docs)}")
first_chunk = langchain_docs[0]
content = first_chunk["text"]
print(f"--- Перший чанк ({len(content)} символів) ---")
print(content.strip())
print(f"Метадані: {first_chunk['metadata']}")

📄 Total chunks created: 225
--- Перший чанк (731 символів) ---
LangChain
Developer Harrison Chase
Initial release October 2022
Stable release0.1.16[1] / 11 April 2024
Written in Python and JavaScript
Type Software framework for largelanguage model applicationdevelopment
License MIT License
Website LangChain.com (https://langchain.com/)
Repository github.com/langchain-ai/langchain (https://github.com/langchain-ai/langchain)
LangChain
Free and open-source softwareportal
LangChain is a software framework that helps facilitate the integration of large language models(LLMs) into applications. As a language model integration framework, LangChain's use-caseslargely overlap with those of language models in general, including document analysis andsummarization, chatbots, and code analysis.[2]
Метадані: {'page_label': '1', 'file_name': 'langchain.pdf', 'file_path': 'c:\\Users\\User\\Desktop\\robotdream-course\\agents\\homework-lesson-5\\data\\langchain.pdf', 'file_type': 'application/pdf', 'file

In [ ]:
# Embedding - text-embedding-3-small
from config import Settings
settings = Settings()

from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

embeddings = OpenAIEmbeddings(
    api_key=settings.api_key.get_secret_value(),
    model=settings.embedding_model
)

text1 = langchain_docs[0]["text"]     
text2 = langchain_docs[1]["text"]     
text_last = langchain_docs[-1]["text"] 

texts = [text1, text2, text_last]

vectors = embeddings.embed_documents(texts)

vec1 = np.array(vectors[0]).reshape(1, -1)
vec2 = np.array(vectors[1]).reshape(1, -1)
vec_last = np.array(vectors[2]).reshape(1, -1)

sim_1_2 = cosine_similarity(vec1, vec2)[0][0]
sim_1_last = cosine_similarity(vec1, vec_last)[0][0]

print(f"📐 Подібність (1-й та 2-й чанки): {sim_1_2:.4f}")
print(f"📐 Подібність (1-й та останній чанки): {sim_1_last:.4f}")

2026-03-20 11:13:39,312 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


📐 Подібність (1-й та 2-й чанки): 0.7954
📐 Подібність (1-й та останній чанки): 0.2596


In [39]:
# Query Engine - Build FAISS Index 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_community.vectorstores import FAISS

texts = [doc["text"] for doc in langchain_docs]
metadatas = [doc["metadata"] for doc in langchain_docs]

vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
print(f"🗃️  FAISS index built!")

vectorstore.save_local("./faiss_index")
print("💾 Index saved to ./faiss_index/")

# пошук 3 найближчі за змістом чанки
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(api_key=settings.api_key.get_secret_value(),model="gpt-5.2", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question based on the provided context.\n\nContext:\n{context}"),
    ("human", "{input}"),
])
question_answer_chain = create_stuff_documents_chain(llm, prompt) #склеює документи в один контекст для відповіді
qa_chain = create_retrieval_chain(retriever, question_answer_chain) #створює ланцюжок, який спочатку виконує пошук, а потім передає знайдені документи в ланцюжок відповіді

print("\n🔍 Testing query: 'What is retrieval-augmented generation?'")
result = qa_chain.invoke({"input": "What is retrieval-augmented generation?"})
print(f"\n📝 Answer: {result['answer']}")
print(f"\n📚 Sources:")
for doc in result['context']:
    print(f"   - Page {doc.metadata.get('page_label', '?')}| {doc.metadata.get('file_name', '?')}")

2026-03-20 12:28:30,195 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-03-20 12:28:30,930 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


🗃️  FAISS index built!
💾 Index saved to ./faiss_index/

🔍 Testing query: 'What is retrieval-augmented generation?'


2026-03-20 12:28:33,454 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



📝 Answer: Retrieval-augmented generation (RAG) is a technique for large language models where the model **first retrieves relevant information from external sources** (such as databases, uploaded documents, or web content) and then **uses that retrieved text along with the user’s query to generate an answer**.

By supplementing the model’s fixed training data with up-to-date or domain-specific documents, RAG can produce responses grounded in authoritative or internal sources—often without needing to retrain the model when new information becomes available.

📚 Sources:
   - Page 1| retrieval-augmented-generation.pdf
   - Page 6| retrieval-augmented-generation.pdf
   - Page 2| retrieval-augmented-generation.pdf


In [37]:
# - **Збереження індексу** — індекс повинен зберігатися на диск і перезавантажуватися без повторного embedding


from langchain_community.vectorstores import FAISS
import os

texts = [doc["text"] for doc in langchain_docs]
metadatas = [doc["metadata"] for doc in langchain_docs]

INDEX_PATH = "./faiss_index"

if os.path.exists(INDEX_PATH):
    # Завантажуємо існуючий індекс — без повторного embedding
    vectorstore = FAISS.load_local(INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
    print("📂 Index loaded from disk!")
else:
    # Будуємо індекс вперше
    vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
    vectorstore.save_local(INDEX_PATH)
    print("🗃️  FAISS index built and saved!")


📂 Index loaded from disk!
